# ask

> answer from the vault, with citations back into it

In [ ]:
#| default_exp ask

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Everything about models is [rishi](https://github.com/vedicreader/rishi)'s. `Chat(model)` picks the
backend from the *shape* of the id — `litert-community/…` or `.litertlm` to LiteRT, `.gguf` to
llama.cpp, `mlx-community/…` to MLX, a hosted name like `claude-sonnet-5` to fastllm, or an explicit
`mlx/…` prefix — raises rather than guessing when the id says nothing, and falls back to a small
local default when given none. So there is no model table here and nothing to resolve: `$VISHALAKSHI_MODEL`
is the only knob this package adds, and `chat=` is how you reach the rest of rishi's constructor.

The numbering `mk_prompt` writes is the contract with the model: `[n]` in the answer maps to
`ctx.results[n-1]`, which is what makes an answer checkable against the vault rather than merely
plausible.

Thinking is rishi's too — every backend puts it in `channels.thought`, which is what `thought(resp)`
reads. `split_reasoning` exists for the one case rishi's own `split_think` cannot see: MLX chat
templates prefill the `<think>` opener, so a Qwen reply arrives with a *closing* `</think>` and
nothing before it. Leaving that in the answer would put citations there that the answer does not
actually make.

In [ ]:
#| export
import os, re, warnings
from contextlib import contextmanager
from fastcore.all import AttrDict, L, patch
from rishi.core import Chat, is_ctx_error, resp_text, split_think, thought
from vishalakshi.core import Vault, tidy_bc

In [ ]:
#| export
VAULT_SP = """You answer questions from a personal research vault.

You are given numbered sections retrieved from the user's own corpus — papers, web pages,
transcripts, files and their own notes. Answer only from those sections.

Rules:
- Cite every claim with the bracketed number of the section it came from, like [2]. A sentence
  drawing on two sections cites both.
- If the sections do not answer the question, say exactly what is missing rather than filling the
  gap from memory. A vault that admits a hole is useful; one that guesses is not.
- Sections marked RELATED were reached by association, not by matching the question. Use them for
  context or to point somewhere worth reading next, and say so when you do.
- Prefer the user's own notes when they conflict with a source, and flag the disagreement."""

dflt_model = os.getenv('VISHALAKSHI_MODEL')

CHAT = Chat   # what `new_chat` builds; `use_chat` swaps it, and that is the whole seam

def new_chat(model:str=None,   # an id, a path, `mlx/…`; None -> $VISHALAKSHI_MODEL
             **kw              # anything else rishi's `Chat` takes: sp, temp, runtime, think, …
):
    'The one place a chat is built — a fresh one per call, so there is no conversation to keep fresh.'
    return CHAT(model or dflt_model, **kw)

def is_stock_chat() -> bool:
    "Is `new_chat` still building rishi's own `Chat`? False while `use_chat` has something else in."
    return CHAT is Chat

@contextmanager
def use_chat(f):
    """Build chats with `f` for the duration, instead of `rishi.Chat`.

    The seam the recording harness needs, now that there is no `chat=` argument to swap: the
    notebooks replay `CachedChat`'s recorded replies through it, with no weights and no network."""
    global CHAT
    old, CHAT = CHAT, f
    try: yield
    finally: CHAT = old


In [ ]:
#| hide
# a stand-in that records what the constructor was handed: the only question here is which id and
# which settings arrived, and no backend is involved in answering it
class _Rec:
    def __init__(self, model=None, **kw): self.model, self.kw, self.hist = model, kw, []

with use_chat(_Rec):
    test_eq(new_chat('a').model, 'a')                                    # the id named
    test_eq(new_chat().model, dflt_model)                                # none -> $VISHALAKSHI_MODEL
    test_eq(new_chat('a', temp=0, sp='hi').kw, dict(temp=0, sp='hi'))    # the rest, straight through
test_eq(CHAT, Chat)                                                      # ...and the swap is undone

In [ ]:
#| export
def mk_prompt(question:str,        # what you want to know
              ctx,                 # AttrDict from Vault.context()
              max_chars:int=4000,  # chars kept per section
              related:bool=True,   # include the associative leg
              note:str='',         # a line about the sections, before the question
) -> str:
    'The user turn: the numbered sections, then the question.'
    def sec(i, r):
        pg = f', pages {r.pages[0]}–{r.pages[1]}' if r.pages and r.pages[0] is not None else ''
        return f"[{i}] {tidy_bc(r.breadcrumb)}\n(source: {r.filename or r.doc_id}{pg})\n\n{(r.text or '')[:max_chars]}"
    parts = L(sec(i, r) for i, r in enumerate(ctx.results, 1))
    if related and ctx.related:
        parts.append('RELATED — not retrieved by the question, but connected to what was:\n' +
                     '\n'.join(f'- {tidy_bc(r.breadcrumb)} (reached by {r.via})' for r in ctx.related))
    if not parts: return f'The vault returned nothing for this question.\n\nQuestion: {question}'
    return '\n\n---\n\n'.join(parts) + f'\n\n---\n\n{note}Question: {question}'

def split_reasoning(text:str) -> tuple:
    "`(answer, thinking)` — rishi's `split_think`, plus the *closing*-only tag an MLX prefill leaves."
    text, think = split_think(text)
    if '</think>' in text:
        pre, _, text = text.partition('</think>')
        think = '\n'.join(L(think, pre.strip()).filter())
    return text.strip(), think

def cited(answer:str, results) -> L:
    'The sections an answer actually cited, in citation order — the audit trail for a claim.'
    ns = dict.fromkeys(int(m) for m in re.findall(r'\[(\d+)\]', answer or ''))
    def one(n, r): return dict(n=n, node_id=r.node_id, title=r.title, source=r.filename,
                               breadcrumb=tidy_bc(r.breadcrumb), doc_id=r.doc_id)
    return L(one(n, results[n-1]) for n in ns if 0 < n <= len(results))

### Answering with citations

`ask` is one call over four backends, because it does not choose one: `chat` defaults to `rishi.Chat`
and gets the model id as rishi wrote it. Pass `chat=partial(Chat, runtime='llama', temp=0)` — or any
callable that builds a chat — when you want anything rishi's constructor takes. `mk_chat` settles the
rest: the id and the settings named at the call site beat the ones the partial carries, which beat
`$VISHALAKSHI_MODEL`. Each call builds its own chat, so there is no conversation to keep fresh.

`ref` is the other half. With none, the question is answered from what retrieval returns. With one or
more documents named, *those* are sections `[1..n]` in full and a few retrieved sections follow them.
Naming several is the point: "what does this module do that `core.py` does not" cannot be answered
from one file, and retrieval will not reliably put the other one in front of the model — so
`ref=['../vishalakshi/extract.py', '../vishalakshi/core.py']` puts both there. `doc_chars` is the
budget for the named documents *together*, shared between them, because a window is a total.

Either way the citation contract is the same, so `cited` resolves `[n]` back to a `node_id` you can
`read()`, and `context` is kept so you can inspect what the model was and was not shown. A cited
section whose `node_id` is `None` came from the code leg — its citation is a `path:line` on disk
rather than something `read()` can open.

The retry is where a local model's real ceiling shows. A prompt too long for the window comes back
from LiteRT as an opaque `send_message failed` that `is_ctx_error` cannot recognise, so `ask` catches
it and asks again with less. It has to ask on a **new chat**: the failed turn is still in the old
conversation's KV cache, and `hist = []` clears rishi's history without freeing it — so a retry into
the same chat is the same overflow with a shorter prompt on top of it. Measured on
`gemma-4-E2B-it-litert-lm`: 1686 tokens answers, 4515 fails.

`schema` turns the answer from prose into a dict of the fields you named. That is the same machinery
`extract` uses, so `ask(q, ref=…, schema='total:float')` and `extract(ref, schema=…)` differ only in
who writes the instruction.

In [ ]:
#| export
def doc_note(n:int) -> str:
    'The line that says which of the numbered sections are the documents being asked about.'
    which = '[1] is the document' if n == 1 else f'[1]–[{n}] are the documents'
    return f'{which} being asked about; the rest is context from elsewhere in the vault.\n\n'

@patch
def doc_context(self:Vault,
                ref,                  # a doc_id, source, title or path — or a list of them
                question:str,         # what the other sections are retrieved against
                related:int=3,        # sections from the *rest* of the vault to add
                max_chars:int=12000,  # chars of the named documents, shared out between them
) -> AttrDict:
    'The documents `ref` names as sections [1..n], with a few sections from elsewhere behind them.'
    refs = L(ref if isinstance(ref, (list, tuple, L)) else [ref])
    # `max_chars` is the budget for the named documents *together*. Three files at a per-file cap is
    # how a question that names three files blows the window of the model meant to answer it.
    docs = refs.map(lambda r: self.document(r, max_chars=max(1, max_chars//len(refs))))
    res = docs.map(lambda d: AttrDict(node_id=f'{d.doc_id}#0' if d.doc_id else '', title=d.title,
                                      doc_id=d.doc_id, breadcrumb=d.title, filename=d.source,
                                      pages=None, text=d.text))
    own, cap = set(docs.attrgot('doc_id')) - {None}, len(docs) + related
    for s in (self.sections(question, limit=related*2) if related else ()):
        if s['node_id'].split('#')[0] in own or len(res) >= cap: continue
        res.append(AttrDict(node_id=s['node_id'], title=s['title'], doc_id=s['node_id'].split('#')[0],
                            breadcrumb=s['breadcrumb'], filename=None, pages=s['pages'],
                            text='\n\n'.join(s['snippets'])))
    return AttrDict(results=res, related=L(), encoder=self.enc.note, doc=docs[0], docs=docs,
                    note=doc_note(len(docs)))

@patch
def ask(self:Vault,
        question:str,          # what you want to know
        ref=None,              # documents to ask about: a doc_id, source, title or path, or a list of them
        schema=None,           # answer as this shape instead of prose: a SCHEMAS key, dataclass, or 'field:type' spec
        model:str=None,        # an id, a path, `mlx/…`; None -> $VISHALAKSHI_MODEL
        chat_kw:dict=None,     # anything else rishi's `Chat` takes: temp, runtime, think, …
        sections:int=4,        # operative sections retrieved
        related:int=6,         # associative sections offered as leads
        kind:str=None,         # restrict retrieval to one or more KINDS
        code:int=None,         # code sections to add; None -> 4 if kosha has indexed the repo
        dir:str=None,          # repo for the code legs; None -> the cwd repo
        max_chars:int=1500,    # chars of each retrieved section shown to the model
        doc_chars:int=8000,    # chars of `ref`'s documents shown to the model, shared between them
        sp:str=VAULT_SP,       # system prompt
        **kw                   # forwarded to Vault.context
) -> AttrDict:
    'Answer with citations back into the vault — about the documents `ref` names, when it names any.'
    mk = lambda: new_chat(model, sp=sp, **(chat_kw or {}))
    if ref is None:
        ctx, note, mc = self.context(question, sections=sections, related=related, kind=kind, code=code,
                                     dir=dir, **kw), '', max_chars
    else:
        ctx = self.doc_context(ref, question, related=related, max_chars=doc_chars)
        note, mc = ctx.note, doc_chars
    prompt = mk_prompt(question, ctx, max_chars=mc, related=bool(related), note=note)
    ch = mk()
    d = ctx.get('doc')
    out = AttrDict(question=question, model=model or dflt_model, runtime=ch.runtime, context=ctx,
                   encoder=self.enc.note, prompt=prompt, answer=None, thinking='', cited=L(),
                   schema=None, fields=None,
                   **(dict(doc_id=d.doc_id, title=d.title, source=d.source, origin=d.origin,
                           chars=sum(len(x.text) for x in ctx.docs),
                           truncated=any(ctx.docs.attrgot('truncated'))) if d else {}))
    if schema is not None:
        # the shapes live in `extract`, with the doctypes that pick them
        from vishalakshi.extract import as_schema, structured
        sch = as_schema(schema, name='Answer', doc=f'The answer to: {question}')
        out.schema, out.fields = sch.__name__, structured(ch, prompt, sch, sp=sp)
    else:
        try: res = ch(prompt)
        except Exception as e:
            # only a failed *send* is worth answering with less of the vault. Retrying on anything
            # at all meant a cache miss, a refused download or a missing extra came back as a
            # quietly degraded two-section answer, which reads exactly like a real one.
            #
            # `is_ctx_error` is not enough by itself: LiteRT reports an overflow as
            # `litert_lm_conversation_send_message failed`, which matches none of rishi's markers,
            # and its `ctx_limit` is None unless a caller sets one, so `pct_full` cannot adjudicate
            # either. What is left is the exception *type* — a backend that tried and failed raises
            # RuntimeError or ValueError; a KeyError or an ImportError never got that far.
            if not (is_ctx_error(ch, e) or isinstance(e, (RuntimeError, ValueError))): raise
            warnings.warn(f'{type(e).__name__} on a {len(prompt)}-char prompt ({e}) — retrying with less '
                          f'context. Lower `sections`/`doc_chars`/`max_chars`, or use a model with a '
                          f'bigger window.')
            ctx.results, ctx.related = ctx.results[:2], L()
            out.prompt = prompt = mk_prompt(question, ctx, max_chars=mc//3, related=False, note=note)
            # a *new* chat, not `hist = []`: the failed turn is still in the old conversation's KV
            # cache, so re-sending into it is the same overflow again with a shorter prompt on top.
            # `new_chat` builds one from the id and the kwargs, which beats reaching for the
            # backend's own `_recreate_conv`
            ch = mk()
            res = ch(prompt)
        # rishi already separates thinking into `channels.thought` on every backend; `split_reasoning`
        # is only for the tags a chat template left in the text itself
        out.answer, out.thinking = split_reasoning(resp_text(res))
        out.answer, out.thinking = out.answer.strip(), out.thinking or thought(res)
        out.cited = cited(out.answer, ctx.results)
    out.usage = getattr(ch, 'use', None)
    return out

@patch
def explain(self:Vault, node_id:str, model:str=None, chat_kw:dict=None, max_chars:int=6000,
            sp:str=VAULT_SP) -> AttrDict:
    'Have a model explain one section in the context of what the vault connects it to.'
    sec, rel = self.read(node_id, max_chars=max_chars), self.related(node_id, limit=6)
    ch = new_chat(model, sp=sp, **(chat_kw or {}))
    prompt = (f"Section: {sec.get('title','')}\n\n{sec.get('text','')}\n\nOther sections in the vault "
              f"that read like it:\n" + '\n'.join(f"- {r['breadcrumb']}" for r in rel) +
              "\n\nExplain this section, then say what the related sections add or contradict.")
    res = ch(prompt)
    answer, thinking = split_reasoning(resp_text(res))
    return AttrDict(node_id=node_id, answer=answer.strip(), thinking=thinking or thought(res),
                    section=sec, related=rel)

### Recording what a model said

A stubbed model tests the plumbing and hides the model: a stub that always returns a well-formed
answer cannot tell you that gemma-4-E2B emits a tool call LiteRT itself rejects. `CachedChat` is the
other approach — a real `rishi.Chat` whose replies are recorded to disk, so the notebooks exercise
real prompts against real replies and CI replays them with no weights, no network and no API key.

It is a `chat=` callback like any other, so nothing in `ask` or `extract` knows about it. Two
properties are what make it worth having in the library rather than in a test file: a replay never
constructs an engine, so a cache hit costs nothing and can never start a download; and a *failure*
is a recorded reply too — the LiteRT tool-call rejection is in the cache, which is how the fallback
in `extract` stays tested. A miss raises unless `$VISHALAKSHI_RECORD_CHAT` is set, so a run in CI
can only ever replay.

In [ ]:
#| export
CHAT_CACHE = 'chatcache'   # a diskcache directory; the one under nbs/ is committed, for CI
class CachedChat:
    """A `rishi.Chat` whose replies are recorded to disk and replayed on a second ask.

    A replay never builds an engine, so it costs nothing and can never start a download. A miss
    needs `$VISHALAKSHI_RECORD_CHAT` (or `record=True`) — otherwise it raises rather than quietly
    reaching for a model. An exception is recorded like any other reply, because a backend that
    rejects its own tool call is exactly what the code around it has to handle."""
    def __init__(self,
                 model:str=None,   # anything rishi takes; None -> $VISHALAKSHI_MODEL
                 path:str=None,    # the diskcache directory; None -> CHAT_CACHE
                 record:bool=None, # allow a miss to reach a real model; None -> $VISHALAKSHI_RECORD_CHAT
                 sp:str='',        # system prompt, part of the key
                 **kw              # forwarded to `rishi.Chat` on a miss
    ):
        from diskcache import Cache
        self.model, self.sp, self.kw = model or dflt_model, sp, kw
        self.cache = Cache(str(path or CHAT_CACHE))
        self.record = bool(os.getenv('VISHALAKSHI_RECORD_CHAT')) if record is None else record
        self._chat, self.hist, self.use = None, [], None

    @property
    def chat(self):
        'The real chat, built only when something actually has to be asked.'
        if self._chat is None: self._chat = Chat(self.model, sp=self.sp, **self.kw)
        return self._chat
    @property
    def runtime(self):
        from rishi.core import resolve_runtime
        return resolve_runtime(self.model)[0]

    def _ask(self, key:str, f):
        'Replay `key`, else run `f()` and record what it did — including how it failed.'
        if key in self.cache:
            kind, val = self.cache[key]
            if kind == 'exc': raise RuntimeError(val)
            return val
        if not self.record: raise KeyError(
            f'no recorded reply for {key[:120]}… — set VISHALAKSHI_RECORD_CHAT=1 and re-run to record it')
        try: val = f()
        except Exception as e:
            self.cache[key] = ('exc', f'{type(e).__name__}: {e}'); raise
        self.cache[key] = ('ok', val)
        return val

    def __call__(self, prompt, **kw):
        return self._ask(f'{self.model}|call|{self.sp}|{prompt}', lambda: dict(self.chat(prompt, **kw)))
    def classify(self, text, labels, sp=None):
        return self._ask(f'{self.model}|classify|{sp}|{",".join(labels)}|{text}',
                         lambda: self.chat.classify(text, labels, sp=sp))
    def structured(self, prompt, schema, sp=None):
        # the reply is stored as a dict, not the object: a `dyn_schema` class cannot be pickled
        from dataclasses import asdict, fields, is_dataclass
        flds = [f.name for f in fields(schema)]
        d = self._ask(f'{self.model}|structured|{sp}|{schema.__name__}{flds}|{prompt}',
                      lambda: (lambda o: asdict(o) if is_dataclass(o) else dict(o))(
                          self.chat.structured(prompt, schema, sp=sp)))
        return schema(**d)
    def close(self):
        if self._chat is not None: self._chat.close(); self._chat = None

In [ ]:
#| hide
# A replay answers from disk and never constructs an engine — which is what lets `06_extract` test
# the model legs in CI, and what stops a miss from quietly starting a multi-gigabyte download.
from tempfile import mkdtemp
_cc = CachedChat('litert/some-model', path=mkdtemp())
test_fail(lambda: _cc('hello'), contains='VISHALAKSHI_RECORD_CHAT')   # a miss cannot reach a model
_cc.cache[f'{_cc.model}|call|{_cc.sp}|hello'] = (
    'ok', dict(role='assistant', content=[dict(type='text', text='hi [1]')]))
test_eq(resp_text(_cc('hello')), 'hi [1]')
# a recorded failure is replayed as a failure: LiteRT rejecting its own tool call is a reply too,
# and the code around it is what has to cope
_cc.cache[f'{_cc.model}|call|{_cc.sp}|boom'] = ('exc', 'RuntimeError: litert_lm_conversation_send_message failed')
test_fail(lambda: _cc('boom'), contains='send_message failed')
test_eq(_cc.runtime, 'litert')          # rishi reads that off the id; no engine needed for it either
assert _cc._chat is None, 'a replay must not build a chat'

The defaults are the product claim: a few pointed sections, not the whole vault. 6 sections at
4000 chars was ~18k characters — past a 2B model's window, and reported by LiteRT as an opaque
send failure that `is_ctx_error` cannot recognise, so `ask` has to notice the size itself.

In [ ]:
#| hide
_v = Vault(':memory:', offline=True)
for i in range(8):
    _v.add(f'# Doc {i}\n\n## Fusion\n\n' + ('rank fusion over legs that share no vector space. ' * 120),
           f'doc {i}', kind='note')
import inspect
_p = inspect.signature(Vault.ask).parameters
test_eq((_p['sections'].default, _p['max_chars'].default), (4, 1500))   # the claim, pinned
test_eq(_p['chat_kw'].default, None)                                   # rishi's constructor, reached by name
# and `doc_chars` is not a round number: measured on gemma-4-E2B through this exact path, 8000 chars
# (3533 tokens) answers on the first send and 10000 (4265) needs the retry. The default is the
# largest that does not pay for a failed prefill on the model the package defaults to.
test_eq(_p['doc_chars'].default, 8000)

# `max_chars` is the lever: four page-long sections are what used to overflow a small window
_long = L(AttrDict(breadcrumb=f'doc {i} › Fusion', filename=f'd{i}.md', doc_id=f'd{i}', pages=None,
                   text='rank fusion over legs that share no vector space. ' * 200) for i in range(4))
_ctx = AttrDict(results=_long, related=L())
_small, _big = (len(mk_prompt('rank fusion', _ctx, max_chars=n)) for n in (1500, 4000))
test_eq((_small // 1000, _big // 1000), (6, 16))                        # 6k pointed vs 16k dumped

# A prompt past the window comes back from LiteRT as an opaque send failure, so `ask` retries with
# less — on a *new* chat. `hist = []` clears rishi's history but not the engine conversation the
# failed turn is still sitting in, so a retry into the same chat overflows exactly as before.
class _Boom:
    'A backend whose first send fails the way LiteRT does. One per chat, so re-use is visible.'
    runtime, use = 'litert', None
    made, seen = [], []
    def __init__(self): self.hist = []; _Boom.made.append(self)
    def __call__(self, p):
        _Boom.seen.append((id(self), len(p)))
        if len(_Boom.seen) == 1: raise RuntimeError('litert_lm_conversation_send_message failed')
        return dict(role='assistant', content=[dict(type='text', text='RRF fuses ranks [1].')])
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    with use_chat(lambda *a, **kw: _Boom()): _r = _v.ask('rank fusion', max_chars=4000)
test_eq(len(_Boom.seen), 2)                              # one retry, not a loop
assert _Boom.seen[1][1] < _Boom.seen[0][1], _Boom.seen   # the second prompt really is smaller
assert _Boom.seen[1][0] != _Boom.seen[0][0], 'the retry must go to a new chat, not the failed one'
test_eq(len(_Boom.made), 2)
assert any('retrying with less context' in str(x.message) for x in w), [str(x.message) for x in w]
test_eq((_r.answer, _r.runtime), ('RRF fuses ranks [1].', 'litert'))   # read off the chat that answered
test_eq(_r.prompt, mk_prompt('rank fusion', _r.context, max_chars=4000//3, related=False))

# ...but only a failed *send* buys the retry. A `CachedChat` miss raises KeyError, and answering it
# with two sections and a warning would hide the one thing the recording harness needs to say.
class _Missing:
    runtime, use, hist = 'litert', None, []
    def __call__(self, p): raise KeyError('no recorded reply for gemma|call|…')
with ExceptionExpected(KeyError), use_chat(lambda *a, **kw: _Missing()):
    _v.ask('rank fusion', max_chars=4000)


## Try it

Retrieval needs no model; only the answering step does. `mk_prompt` is the whole contract, so it is
worth looking at what the model actually sees.

In [ ]:
v = Vault(':memory:')
v.note('Late chunking beats naive chunking because context survives the split.')
print(mk_prompt('why late chunking?', v.context('late chunking'))[:400])

[1] Late chunking beats naive chunking because context survives the split.
(source: note:a20ce4c7b9ac, pages 0–0)

Late chunking beats naive chunking because context survives the split.

---

[2] repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/core.py:112
(source: /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/core.py:112)

def __repr__(self):
        s = self.stats()
     


In [ ]:
# a document to ask about becomes section [1] in full, and the rest of the vault follows it
_dc = _v.doc_context('doc 3', 'rank fusion', related=2)
test_eq(_dc.results[0].breadcrumb, 'doc 3')
test_eq(_dc.results[0].text, _v.document('doc 3').text)             # the whole document, not a chunk
assert all(r.doc_id != _dc.doc.doc_id for r in _dc.results[1:])     # never the document itself twice
test_eq(len(_v.doc_context('doc 3', 'rank fusion', max_chars=200).results[0].text), 200)
assert mk_prompt('rank fusion', _dc, note=_dc.note).endswith(_dc.note + 'Question: rank fusion')

# several documents: each is a numbered section, none of them comes back as context, and the
# `max_chars` budget is shared rather than paid per file — a window is a total
_dc2 = _v.doc_context(['doc 3', 'doc 5'], 'rank fusion', related=2, max_chars=600)
test_eq(_dc2.results.attrgot('breadcrumb')[:2], ['doc 3', 'doc 5'])
test_eq([len(r.text) for r in _dc2.results[:2]], [300, 300])
assert all(r.doc_id not in set(_dc2.docs.attrgot('doc_id')) for r in _dc2.results[2:])
test_eq(len(_dc2.results), 4)                                       # 2 named + related=2
assert '[1]–[2] are the documents' in _dc2.note, _dc2.note
test_eq(doc_note(1)[:19], '[1] is the document')

In [ ]:
res = L([AttrDict(node_id='d#1', title='A', breadcrumb='A › B', filename='f', doc_id='d')])
test_eq(cited('as [1] shows, and again [1], but not [9]', res).attrgot('node_id'), ['d#1'])

In [ ]:
# reasoning arrives three ways: a full pair, an opener cut off at the cap, and — from an MLX chat
# template that prefills the opener — a bare closing tag with the reasoning in front of it
test_eq(split_reasoning('<think>weighing [1]</think>\n\nRRF fuses ranks [2].'),
        ('RRF fuses ranks [2].', 'weighing [1]'))
test_eq(split_reasoning('weighing [1]</think>\n\nRRF fuses ranks [2].'),
        ('RRF fuses ranks [2].', 'weighing [1]'))
test_eq(split_reasoning('RRF fuses ranks [2].'), ('RRF fuses ranks [2].', ''))
# a section the reasoning weighed and dropped must not come back as a citation
test_eq(cited(split_reasoning('cites [1]</think> cites nothing')[0], res), [])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()